# Unidad 3: Transformación de datos y Feature Engineering

En esta clase vamos a continuar trabajando con el dataset **`satis_clientes.csv`**.

## Objetivos de la clase

- Normalizar y renombrar títulos de columnas
- Transformar tipos de datos
- Realizar selecciones y filtros con Pandas
- Aplicar transformaciones sobre los datos
- Crear nuevas variables mediante **Feature Engineering**

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np

## 1. Carga del dataset

Las transformaciones de este Notebook deberían hacerse sobre el dataframe limpio `airbnb_jr_clean.csv` que construimos en la clase pasada, sin duplicados ni nulos.

Pero para entender los problemas de aplicar transformaciones sin limpiar antes los datos, vamos a comenzar usando el dataset con duplicados y nulos.

Se sugiere, una vez entendido todo el Notebook, ajustar el dataset.

In [ ]:
# pd.read_csv recibe la url del dataset
df = pd.read_csv('https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv')
print("Dataset cargado exitosamente! \nDimensiones:", df.shape)

In [ ]:
# Link generado al compartir el archivo
# path = "airbnb_jr_clean.csv"

# Lo importamos usando el método read_csv() con la ruta path
# df = pd.read_csv(path)
# df.shape

## 2. Exploración inicial

Antes de transformar los datos, conviene revisar su estructura general.

In [ ]:
# Mostrar las variables y su tipo de dato
df.dtypes

# Eventualmente usar info()

## 3. Normalización y renombrado de títulos de columnas

En muchos datasets, los nombres de columnas pueden tener:
- espacios innecesarios
- mayúsculas y minúsculas mezcladas
- tildes o caracteres especiales
- nombres poco prácticos para programar

Por eso, una buena práctica es **normalizar** los títulos para dejarlos prolijos y consistentes.

In [ ]:
# Normalizamos nombres de columnas:
# - quitamos espacios laterales
# - pasamos a minúsculas
# - reemplazamos espacios por guiones bajos
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
)

df.columns

**Consultar con la IA** opciones para mejorar la normalización, por ejemplo si quisiera quitar tildes.

A veces además de normalizar necesitamos **renombrar** columnas para que sean más claras o más cómodas de usar.

In [ ]:
# Renombrado manual de columnas (ajustar según necesidad del dataset)
df = df.rename(columns={
    'neighbourhood': 'barrio'
})

df.columns

## 4. Transformación de tipos de datos

Pandas puede interpretar una columna con un tipo de dato que no sea el más conveniente para analizar.

Por ejemplo:
- una fecha puede venir como texto
- un precio puede venir como string

Corregir los tipos de datos mejora la calidad del análisis y evita errores posteriores.

### 4.1 Convertir dtype `object` a `string`

Recordemos que al momento de importar un dataset, Python asume el tipo de dato. 

En Pandas, una columna con `dtype="object"` no necesariamente contiene únicamente strings. Puede contener strings, números, None u objetos mezclados.

Por eso puede ser útil convertir explícitamente a string cuando necesitás aplicar operaciones de texto de forma consistente.


Vamos a usar el método `astype`, se sugiere consultar la [documentación de astype aquí.](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

In [ ]:
# Ejemplo: asegurar que barrio sea texto
df['barrio'] = df['barrio'].astype('string')
df["barrio"].dtype

### 4.2 Convertir columnas a `int` o `float`

Podríamos usar el mismo método `astype` pero con el argumento apropiado:

In [ ]:
# Convertir bedrooms de float a int
df['bedrooms'] = df['bedrooms'].astype('int')
df['bedrooms'].dtype

Sin embargo, en caso que la cadena de caracteres tenga valores nulos o no sea convertible, generará un error. 

Para ello se puede usar el atributo `errors`. 
En la documentación, este parámetro admite:
* 'raise' (default) -> Genera una exception
* 'ignore' -> ignora, no convierte

In [ ]:
# Trabajamos con una copia para explorar las opciones:
df_test = df.copy()
df_test['bedrooms'] = df_test['bedrooms'].astype('int', errors="ignore")
df_test['bedrooms'].dtype

# Auxiliares
# df_test.dropna(inplace=True)
# print(f"(Nulos para bedrooms: {df_test['bedrooms'].isnull().sum()}")

### 4.3 Convertir columnas numéricas con `to_numeric()`

Cuando una variable numérica viene como texto, podemos convertirla con `pd.to_numeric()`.

Vamos a usar el método `to_numeric`, se sugiere consultar la [documentación de to_numeric aquí.](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html#pandas.to_numeric)

En la documentación, el parámetro errors admite:
* 'raise' -> Genera una exception
* 'coerce' -> convierte a NaN

Explorar también el parámetro `downcast`

In [ ]:
# Ejemplo: convertir bedrooms a numérico
df_test = df.copy()
df_test = df_test.dropna()
df_test['bedrooms'] = pd.to_numeric(df_test['bedrooms'], errors="raise")
df_test['bedrooms'].dtype

# Auxiliares
# df_test = df_test.dropna()
# print(f"(Nulos para bedrooms: {df_test['bedrooms'].isnull().sum()}")
# df_test['bedrooms'] = pd.to_numeric(df_test['bedrooms'], errors="raise", downcast="integer")



Probar ahora con la variable `price`

In [ ]:
df_test = df.copy()
df_test['price'] = pd.to_numeric(df_test['price'], errors='raise')
df_test['price'].dtype

# df_test = df_test.dropna()
# print(f"(Nulos para price: {df_test['price'].isnull().sum()}")


### 4.4 Convertir fechas con `to_datetime()`

Vamos a usar el método `to_datetime`, se sugiere consultar la [documentación de to_datetime aquí.](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html#pandas.to_datetime)

Este método, posee particularmente, más parámetros.

Además de `errors`, aquí se usa generalmente `format`, para indicar si la fecha esta en formato dd/mm/aaaa o yyyy/mm/dd, por ejemplo.

In [ ]:
# Exploramos algunos registros de last_review
df['last_review'].head()

In [ ]:
# Ejemplo: convertir fecha a formato datetime
df_test = df.copy()
df_test['last_review'] = pd.to_datetime(df_test['last_review'], format="%Y-%m-%d", errors='coerce')
df_test['last_review'].dtype

# Auxiliares
# df_test.dropna(inplace=True)
# print(f"(Nulos para last_review: {df_test['last_review'].isnull().sum()}")

Si el "parse" se hizo correctamente, podemos extraer el `year`, `month`, etc de la siguiente manera:

In [ ]:
# Acceder al year directamente
df_test['last_review'].dt.year.head()

# O al month con:
# df_test['last_review'].dt.month.head()

## 5. Selecciones con Pandas

Las **selecciones** permiten elegir columnas o filas específicas del DataFrame.

In [ ]:
# Seleccionar una sola columna
df['barrio']

In [ ]:
# Seleccionar múltiples columnas
df[['barrio', 'bedrooms', 'price']]

In [ ]:
# Seleccionar un conjunto de filas por índice (asociar con slicing en Python)
df.loc[0:5]

In [ ]:
# Seleccionar un subconjunto de filas y columnas
df.loc[0:4, ['barrio', 'bedrooms', 'price']]

## 6. Transformaciones de datos

Las transformaciones modifican el contenido o la estructura del DataFrame para prepararlo mejor para el análisis.

Las transformaciones pueden sobrescribir los datos de la columna o bien podemos generar una nueva.

### 7.1 Transformar price removiendo el literal $

Vamos a usar el método `replace`, se sugiere consultar la [documentación de replace aquí.](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)

In [ ]:
# Repasemos los dtypes de las variables del dataframe
df.dtypes

Para convertir (cast) price de object a float, debemos primero remover el literal $

In [ ]:
# Usamos replace para remover el literal $
df_test = df.copy()
df_test['price'] = df_test['price'].str.replace("$", "", regex=False)

print(f"df['price'].dtype: {df_test['price'].dtype}")
df_test.head(2)

In [ ]:
# Ahora si deberíamos poder convertir price a float
df_test['price'] = pd.to_numeric(df_test['price'], errors="raise", downcast='float')
print(f"df['price'].dtype: {df_test['price'].dtype}")

# Auxiliares
# print(f"nulos en price: {df_test['price'].isnull().sum()}")
df_test.head(2)

Después de eliminar el literal `$` que sucedió al querer convertir a `float`.

Intentá remover el literal `,` usando `.str.replace(",", "")` ahora que sucede?

**Conclusión**

`pd.to_numeric()` permite convertir una columna a un tipo numérico de forma más flexible y segura que `astype()`. 

El parámetro `errors="coerce"` resulta especialmente útil durante la limpieza y preparación de datos, ya que los valores que no pueden convertirse se reemplazan por NaN en lugar de interrumpir la ejecución con un error.

### 7.3 Transformar cadenas de texto

In [ ]:
# Podemos estandarizar los nombres de la columna barrios del dataframe mediante métodos simples:
df_test = df.copy()
df_test['barrio'] = (
        df_test['barrio']
        .str.strip()
        .str.lower()
)

# otros métodos posibles: upper(), title(), capitalize()

df_test.head()

## 8. Limpieza y transformación

Ahora que exploramos todas las opciones, dejemos el dataframe en codiciones para estudiar ordenamiento y feature engineering

In [ ]:
# Remover registros duplicados
df.drop_duplicates(inplace=True)
print(f"Regitros duplicados: {df.duplicated().sum()}")

# Remover nulos
df.dropna(inplace=True)
nulos = df.isnull().sum()
print(f"Campos con valores nulos: {nulos[nulos > 0].sort_values(ascending=False)}")

In [ ]:
df.dtypes

In [ ]:
# Convertimos bedrooms de float a int
df["bedrooms"] = df["bedrooms"].astype('int', errors="ignore")

# Removemos caracteres de price y convertimos a float
df['price'] = df['price'].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Convertimos last_review a datetime
df["last_review"] = pd.to_datetime(df['last_review'], format="%Y-%m-%d", errors='coerce')

## 9. Ordenar datos

In [ ]:
# Ordenar por calificación de mayor a menor
df.sort_values(by='bedrooms', ascending=False) # no sobrescribe nada

# Explorar alternativas
# df = df.sort_values(by='bedrooms', ascending=True) # sobrescribe
# df.sort_values(by='bedrooms', ascending=True, inplace=True) # inplace=True sobrescribe el mismo dataframe


In [ ]:
# Ordenar por calificación de mayor a menor y por fecha
df.sort_values(by=['bedrooms', 'price'], ascending=False)


## 10. Feature Engineering

El **Feature Engineering** consiste en crear nuevas variables a partir de columnas existentes para enriquecer el análisis.

Estas nuevas variables pueden ayudar a:
- segmentar mejor los datos
- resumir información
- preparar el dataset para modelos o análisis posteriores

### 10.1 Crear una variable binaria

In [ ]:
# Cliente satisfecho: True si la calificación es alta
df['cliente_satisfecho'] = df['review_scores'] >= 4.5
df[['review_scores', 'cliente_satisfecho']].sample(5)

### 10.2 Crear una variable usando Apply

In [ ]:
# Cliente satisfecho: True si la calificación es alta, alternativa usando Apply
df['cliente_estado'] = df['review_scores'].apply(
    lambda x: 'Satisfecho' if x >= 4.5 else 'Insatisfecho'
)
df[['review_scores', 'cliente_satisfecho', 'cliente_estado']].sample(5)

### 10.3 Crear una variable temporal a partir de una fecha

In [ ]:
# Extraer componentes de fecha
df['last_review_year'] = df['last_review'].dt.year
df['last_review_month'] = df['last_review'].dt.month

df[['last_review', 'last_review_year', 'last_review_month']].head()

## 11. Cierre

En esta clase trabajamos sobre un flujo muy habitual en análisis de datos:

1. revisar el dataset
2. normalizar nombres de columnas
3. corregir tipos de datos
4. transformar el DataFrame
5. crear nuevas variables de análisis

Estos pasos son fundamentales para preparar datos antes de hacer análisis más avanzados, visualizaciones o modelos.